# Temporal Cycle-Consistency Learning
## Self-supervised video representations with TCN $\rightarrow$ TCC using `aegean-ai/tcc`

This notebook is designed to read like a **tutorial** and a **course assignment** at the same time.

It focuses on one question:

> Can a self-supervised video representation learn the **latent phase** of a task purely from temporal structure, without robot control, action labels, or transcription?

The answer developed across the two Google Research papers is:

1. **TCN** learns by enforcing *time-based contrastive alignment* under strong synchronization assumptions.
2. **TCC** generalizes this idea by enforcing *temporal cycle-consistency*, which is more robust to variations in execution speed and alignment.

In this notebook you will:

- study the conceptual evolution from **TCN** to **TCC**
- train the **PyTorch rewrite** in `aegean-ai/tcc`
- extract frame embeddings
- visualize trajectories with **PCA**, **t-SNE**, and **UMAP**
- segment action sequences using representation geometry

## Papers

- Sermanet et al., **Time-Contrastive Networks**, 2018  
  https://arxiv.org/abs/1704.06888

- Dwibedi et al., **Temporal Cycle-Consistency Learning**, 2019  
  https://arxiv.org/abs/1904.07846

## Repository used in this notebook

- `https://github.com/aegean-ai/tcc`  
  This notebook assumes the **`main` branch** and the current PyTorch package layout under `src/tcc/`.

## What you should learn

By the end, you should be able to explain why TCN and TCC are related but not identical:

- TCN: **metric alignment with synchronized positives**
- TCC: **structural temporal alignment via cycles**

A useful mental model is:

- TCN says: *"frames at the same time index should be close."*
- TCC says: *"if I map from one sequence to another and back, I should return to the same temporal phase."*

## 1. Theory recap: from TCN to TCC

### 1.1 TCN: contrastive temporal alignment

TCN learns an embedding
$$
z_t = f_\theta(I_t)
$$
so that synchronized frames from different views become neighbors in feature space.

A canonical triplet-style loss is:
$$
\mathcal{L}_{\mathrm{TCN}}
=
\max\left(
0,\;
\|f(I_t^a)-f(I_t^b)\|_2^2
-
\|f(I_t^a)-f(I_{t'}^b)\|_2^2
+
\alpha
\right).
$$

Interpretation:

- **anchor:** frame $I_t^a$
- **positive:** synchronized frame $I_t^b$
- **negative:** mismatched-time frame $I_{t'}^b$

This already encodes an important idea: **time is supervision**.

But TCN assumes that corresponding frames are available at matching time indices, which is a strong assumption.

### 1.2 Why TCN is not enough

Suppose two people perform the same pouring task:

- one moves slowly
- one moves quickly
- one pauses before tilting
- one starts tilting earlier

Then the frame with semantic phase "tilt begins" is **not** guaranteed to occur at the same time index in both videos.

So absolute time matching becomes fragile.

### 1.3 TCC: align temporal structure, not raw clock time

TCC keeps the idea that embeddings should reflect task progression, but replaces hard synchronized matching with **cycle consistency**.

**Conceptual intuition.** Given frame $i$ in sequence $A$, map it to the most corresponding frame in sequence $B$:
$$
j = \arg\min_k \|f(I_i^A)-f(I_k^B)\|.
$$

Then map back from sequence $B$ to sequence $A$:
$$
i' = \arg\min_l \|f(I_j^B)-f(I_l^A)\|.
$$

TCC encourages $i' \approx i$.

**Differentiable training loss.** The hard `argmin` above is not differentiable, so the actual TCC loss replaces it with a **soft nearest-neighbor** formulation. For frame $i$ in sequence $A$, define a soft correspondence distribution over frames in sequence $B$:

$$
\beta_k^{(i)} = \frac{\exp(-\|f(I_i^A) - f(I_k^B)\|^2 / \tau)}{\sum_{k'} \exp(-\|f(I_i^A) - f(I_{k'}^B)\|^2 / \tau)}
$$

where $\tau$ is a temperature parameter. The cycle-back distribution is computed analogously, and the loss is the cross-entropy between the back-mapped distribution and a target concentrated at the original index $i$. This makes the entire cycle differentiable and trainable with standard gradient descent.

Conceptually:

- TCN aligns **absolute timestamps**
- TCC aligns **latent phase structure**

This is why TCC is more appropriate when demonstrations are semantically similar but **temporally warped**.

### 1.4 What the embedding should look like on pouring

If TCC works, then the learned trajectory in embedding space should behave like a latent phase variable:

- early reach frames cluster near other early reach frames
- grasp transitions appear near one another
- tilt and pour form coherent regions
- embeddings from different videos should trace similar temporal paths

That is the premise you will test below.

## 2. How this notebook and the `aegean-ai/tcc` repo work together

This notebook is **not** a standalone script. It is a guided analysis layer that drives the `aegean-ai/tcc` PyTorch package. The repo provides the training loop, model definitions, dataset utilities, and evaluation code. The notebook provides the experimental protocol: configuring runs, extracting embeddings, and visualizing results.

### Two supported environments

| | **Dev container (recommended)** | **Google Colab** |
|---|---|---|
| **GPU** | Local NVIDIA GPU via Docker | Colab T4/A100 runtime |
| **Package manager** | `uv` (pre-installed in container) | `pip` (Colab default) |
| **Setup effort** | `make start` — one command | Clone + pip install in notebook cells |
| **Persistence** | Full local disk | Session-scoped (data lost on disconnect) |
| **Best for** | Full sweep, large runs | Quick experiments, no local GPU |

Choose **one** environment and follow the corresponding setup path in Section 3.

### Workflow overview

```
┌──────────────────────────────────────────────────────────────────┐
│  This notebook (analysis layer)                                  │
│                                                                  │
│  1. Set up environment (dev container OR Colab)                  │
│  2. Prepare the pouring dataset                                  │
│  3. Configure training via tcc.config.get_default_config()       │
│  4. Launch training via tcc.train.train(cfg)                     │
│  5. Load checkpoints via tcc.train.load_checkpoint()             │
│  6. Extract embeddings via tcc.evaluate.get_embeddings_dataset() │
│  7. Visualize and segment (PCA, UMAP, KMeans — notebook code)   │
└──────────────────────────────────────────────────────────────────┘
         │                        ▲
         │  function calls        │  returns tensors,
         ▼                        │  checkpoints, configs
┌──────────────────────────────────────────────────────────────────┐
│  aegean-ai/tcc  (installed as editable package)                  │
│                                                                  │
│  src/tcc/                                                        │
│  ├── config.py          TCCConfig dataclass + get_default_config │
│  ├── train.py           Training loop, checkpoint save/load      │
│  ├── evaluate.py        Embedding extraction, eval metrics       │
│  ├── datasets.py        DataConfig, create_dataset()             │
│  ├── models.py          ResNet backbone + embedding head         │
│  ├── alignment.py       TCC alignment algorithm                  │
│  ├── losses.py          Cycle-consistency loss                   │
│  └── algos/             Algorithm registry (tcc, tcn, sal, …)    │
│                                                                  │
│  configs/                                                        │
│  └── default.yaml       Default hyperparameters                  │
│                                                                  │
│  scripts/                                                        │
│  └── download_pouring_data.sh                                    │
│                                                                  │
│  src/tcc/dataset_preparation/                                    │
│  ├── videos_to_dataset.py    Raw videos → image folders          │
│  ├── images_to_dataset.py    Images → dataset structure          │
│  └── visualize_dataset.py    Inspect prepared data               │
└──────────────────────────────────────────────────────────────────┘
```

### What you modify vs. what you use as-is

| Layer | You modify | You use as-is |
|-------|-----------|---------------|
| **Notebook** | Embedding dimension, iteration count, analysis parameters ($k$, projection method) | Visualization and segmentation code |
| **Repo config** | `model.conv_embedder.embedding_size`, `train.max_iters`, `logdir` | Everything else in `configs/default.yaml` |
| **Repo code** | Nothing — treat as a library | `train.py`, `evaluate.py`, `datasets.py`, `models.py` |

## 3. Environment setup

Choose **one** of the two paths below. Both result in a working `import tcc` with GPU access.

---

### Path A: Dev container (recommended for full assignment)

The repo ships a complete Docker-based development environment with GPU support, `uv`, and VS Code integration.

**Prerequisites:** Docker with NVIDIA Container Toolkit, VS Code with Dev Containers extension.

**Steps:**

1. Clone the repo locally:
   ```bash
   git clone https://github.com/aegean-ai/tcc && cd tcc
   ```
2. Copy the environment file:
   ```bash
   cp .env.example .env
   # Edit .env to add WANDB_API_KEY and/or HF_TOKEN if needed
   ```
3. Open in VS Code → "Reopen in Container" (or run `docker compose up -d` manually).
4. Inside the container, run:
   ```bash
   make start
   ```
   This creates a `.venv` with `uv`, installs the package in editable mode, and registers a Jupyter kernel.
5. Open this notebook in VS Code or JupyterLab (port 8888) and select the **"Python 3 (tcc)"** kernel.

**Key details:**
- Base image: `pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime`
- Package manager: `uv` (not pip) — the Makefile handles all `uv` calls
- Python: whatever 3.11+ is in the container (typically from conda)
- Workspace: `/workspaces/tcc`
- TensorBoard: port 6006

**Installing extra notebook dependencies** (matplotlib, umap-learn, etc.):
```bash
make install-notebooks
```

---

### Path B: Google Colab (quick start, no local GPU needed)

Use this path if you do not have a local GPU or want a fast start. Colab sessions are ephemeral — save checkpoints to Google Drive to avoid losing training results.

**Steps:**

1. In a Colab notebook, enable GPU: **Runtime → Change runtime type → T4 GPU**.
2. Run the clone and install cells below (Section 3.1–3.2).
3. Colab uses `pip` — the `%pip install` commands handle everything.

**Limitations:**
- Session timeout erases all local files. Mount Google Drive for persistence:
  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  # Point EXPERIMENT_ROOT and DATA_ROOT to /content/drive/MyDrive/tcc/
  ```
- Colab's default Python may differ from 3.11 — the package should still install but is only tested on 3.11–3.12.

---

### Python version requirement

The repo requires **Python ≥3.11, <3.13** (`pyproject.toml`). The dev container satisfies this automatically. On Colab, check with `!python --version`.

In [1]:
import sys, platform, os, pathlib

# Detect runtime environment
IN_COLAB = "google.colab" in sys.modules

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("Running in Colab:", IN_COLAB)

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.35
Working directory: /workspaces/tcc
Running in Colab: False


### 3.1 Clone the repository (Colab / Path B only)

If you are using the **dev container** (Path A), skip this — the repo is already your workspace at `/workspaces/tcc`.

In [2]:
if IN_COLAB:
    import subprocess

    REPO_URL = "https://github.com/aegean-ai/tcc"
    REPO_DIR = pathlib.Path("tcc")

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=False)
    else:
        print("Repository already exists:", REPO_DIR)

    print("Repo dir exists:", REPO_DIR.exists())
else:
    print("Skipping clone — running inside dev container.")

Skipping clone — running inside dev container.


### 3.2 Install the package (Colab / Path B only)

If you are using the **dev container** (Path A), skip this — `make start` already installed the package. Run `make install-notebooks` if you need matplotlib/umap-learn.

In [3]:
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./tcc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "matplotlib", "scikit-learn", "umap-learn", "tqdm", "pyyaml"], check=True)

import importlib
try:
    importlib.import_module("tcc")
    print("tcc package is available.")
except ModuleNotFoundError:
    print("tcc not found. Follow the install instructions for your environment (Path A or B).")

tcc package is available.


### 3.3 Quick repository inspection

Verify the repo structure. In the dev container the repo root is `/workspaces/tcc`; on Colab it is the cloned `tcc/` directory.

In [4]:
import os

# Detect environment: dev container vs Colab
if pathlib.Path("/workspaces/tcc/src/tcc").exists():
    REPO_ROOT = pathlib.Path("/workspaces/tcc")
elif IN_COLAB:
    REPO_ROOT = REPO_DIR  # set in the clone cell
else:
    # Fallback: assume we're at the repo root already
    REPO_ROOT = pathlib.Path.cwd()

def walk_top(path, max_depth=2):
    base = os.path.abspath(path)
    label = os.path.basename(base)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth <= max_depth:
            print(root.replace(base, label))
            for f in files[:10]:
                print("   ", f)

walk_top(str(REPO_ROOT), max_depth=2)

tcc
    Makefile
    CLAUDE.md
    uv.lock
    AGENTS.md
    .notebook-target.yml
    README.md
    .env
    docker-compose.yml
    .gitignore
    pyproject.toml
tcc/docs
    architecture.md
tcc/.venv
    .lock
    CACHEDIR.TAG
    .gitignore
    pyvenv.cfg
tcc/.venv/bin
    jupyter-migrate
    numpy-config
    pydoc.bat
    jupyter-run
    jsonschema
    activate.fish
    pyftsubset
    activate.bat
    deactivate.bat
    isympy
tcc/.venv/share
tcc/.venv/lib
tcc/tcc
    Makefile
    CLAUDE.md
    uv.lock
    AGENTS.md
    .notebook-target.yml
    README.md
    docker-compose.yml
    .gitignore
    pyproject.toml
    .env.example
tcc/tcc/docs
    architecture.md
tcc/tcc/scripts
    execute_notebook.py
    download_pouring_data.sh
tcc/tcc/notebooks
    notebook-database.yml
tcc/tcc/src
tcc/tcc/tests
    test_datasets.py
    test_train.py
    test_algos.py
    test_config.py
    test_models.py
    __init__.py
    test_dataset_preparation.py
    test_evaluation.py
    test_losses.py
tcc/t

tcc/.git/branches
tcc/.git/info
    exclude
tcc/.git/hooks
    sendemail-validate.sample
    pre-push.sample
    commit-msg.sample
    prepare-commit-msg.sample
    post-update.sample
    fsmonitor-watchman.sample
    pre-applypatch.sample
    pre-receive.sample
    pre-merge-commit.sample
    applypatch-msg.sample
tcc/.git/refs
    stash
tcc/.git/logs
    HEAD
tcc/configs
    default.yaml
    demo.yaml
tcc/docker
    Dockerfile.torch.dev.gpu
tcc/.devcontainer
    devcontainer.json
tcc/runs_tutorial
tcc/runs_tutorial/pouring_tcc_d32
    config.yaml
tcc/.beads
    metadata.json
    dolt-server.activity
    README.md
    config.yaml
    dolt-monitor.pid
    interactions.jsonl
    .gitignore
    dolt-server.port
tcc/.beads/backup
    dependencies.jsonl
    issues.jsonl
    backup_state.json
    labels.jsonl
    comments.jsonl
    events.jsonl
    config.jsonl
tcc/.beads/hooks
    pre-commit
    prepare-commit-msg
    pre-push
    post-checkout
    post-merge


## 4. Data: the pouring dataset

The multiview pouring dataset is available from two sources:

| Source | URL / Location | What you get |
|--------|---------------|--------------|
| **HuggingFace** | [`sermanet/multiview-pouring`](https://huggingface.co/datasets/sermanet/multiview-pouring) | Raw TFRecord files (requires conversion) |
| **S3 / MinIO lakehouse** | `s3://landing/tcc/pouring/` | Raw TFRecords **and/or** pre-processed image folders |

Set `DATA_SOURCE` in the next cell to choose your download path.

### Expected directory layout

After download and conversion, the dataset root must have this structure:

```
data/pouring_processed/pouring/
├── train/
│   ├── video_001/
│   │   ├── frame_0000.png
│   │   ├── frame_0001.png
│   │   └── ...
│   ├── video_002/
│   │   └── ...
│   └── ...
└── val/
    ├── video_050/
    │   └── ...
    └── ...
```

Each video is a directory of sequentially numbered frames. The PyTorch `create_dataset` function expects this layout — it discovers videos by listing subdirectories under `train/` or `val/`, then loads frames in filename-sorted order.

### S3 bucket layout

The landing bucket mirrors the local layout:

```
s3://landing/tcc/pouring/
├── raw/                          ← TFRecord files (same as HuggingFace)
│   ├── train/*.tfrecord
│   ├── val/*.tfrecord
│   └── test/*.tfrecord
└── processed/pouring/            ← Pre-processed image folders (ready to train)
    ├── train/video_001/...
    └── val/video_050/...
```

In [5]:
DATA_ROOT = pathlib.Path("data")
RAW_POURING_ROOT = DATA_ROOT / "pouring"
PROCESSED_POURING_ROOT = DATA_ROOT / "pouring_processed"

RAW_POURING_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_POURING_ROOT.mkdir(parents=True, exist_ok=True)

print("Raw data dir:", RAW_POURING_ROOT.resolve())
print("Processed data dir:", PROCESSED_POURING_ROOT.resolve())

Raw data dir: /workspaces/tcc/data/pouring
Processed data dir: /workspaces/tcc/data/pouring_processed


In [6]:
# ── Data source selection ──────────────────────────────────────────
# "huggingface" — download raw TFRecords from HuggingFace Hub
# "s3_raw"      — download raw TFRecords from the lakehouse landing bucket
# "s3_processed"— download pre-processed image folders (skip conversion, ready to train)

DATA_SOURCE = "huggingface"

# S3 / MinIO lakehouse settings (used when DATA_SOURCE starts with "s3")
S3_ENDPOINT  = os.environ.get("MINIO_ENDPOINT", "http://192.168.1.26:9000")
S3_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "")
S3_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "")
S3_BUCKET    = "landing"
S3_PREFIX    = "tcc/pouring"
S3_USE_SSL   = S3_ENDPOINT.startswith("https")

print(f"DATA_SOURCE: {DATA_SOURCE}")
if DATA_SOURCE.startswith("s3"):
    print(f"S3 endpoint: {S3_ENDPOINT}")
    print(f"S3 path:     s3://{S3_BUCKET}/{S3_PREFIX}/")
    print(f"Credentials: {'set' if S3_ACCESS_KEY else 'MISSING — check .env'}")

DATA_SOURCE: huggingface


### 4.1 Download from HuggingFace

> **Runs when** `DATA_SOURCE = "huggingface"`. Skip if using S3.

The dataset is hosted at [`sermanet/multiview-pouring`](https://huggingface.co/datasets/sermanet/multiview-pouring) and contains TFRecord files organized into `train/`, `val/`, and `test/` splits.

Use `huggingface_hub.snapshot_download` to download the full dataset. This downloads all files (TFRecords, recombination scripts, README) into a local cache and returns the path. We then symlink or copy into our expected `data/pouring/` directory.

> **Note:** One test file (`whiteorange_to_clear1_real`) was split into two parts due to upload size limits. After downloading, run the provided shell script to recombine it. This only affects the test split — training and validation are ready to use immediately.

In [7]:
if DATA_SOURCE == "huggingface":
    from huggingface_hub import snapshot_download

    hf_cache_path = snapshot_download(
        repo_id="sermanet/multiview-pouring",
        repo_type="dataset",
        local_dir=str(RAW_POURING_ROOT),
    )

    print("Dataset downloaded to:", hf_cache_path)

    for split_dir in sorted(RAW_POURING_ROOT.iterdir()):
        if split_dir.is_dir() and not split_dir.name.startswith("."):
            tfrecords = list(split_dir.glob("*.tfrecord*"))
            print(f"  {split_dir.name}/: {len(tfrecords)} TFRecord file(s)")
else:
    print(f"Skipping HuggingFace download — DATA_SOURCE={DATA_SOURCE!r}")

/workspaces/tcc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 816 files:   0%|          | 0/816 [00:00<?, ?it/s]

Fetching 816 files:  12%|█▏        | 95/816 [00:00<00:00, 936.68it/s]

Fetching 816 files:  23%|██▎       | 189/816 [00:00<00:00, 935.31it/s]

Fetching 816 files:  35%|███▍      | 284/816 [00:00<00:00, 941.68it/s]

...

Fetching 816 files:  71%|███████   | 581/816 [00:00<00:00, 978.17it/s]

Fetching 816 files:  84%|████████▍ | 685/816 [00:00<00:00, 988.99it/s]

Fetching 816 files:  97%|█████████▋| 788/816 [00:00<00:00, 1000.90it/s]

Fetching 816 files: 100%|██████████| 816/816 [00:00<00:00, 979.84it/s] 

Dataset downloaded to: /workspaces/tcc/data/pouring
  labels/: 0 TFRecord file(s)
  tfrecords/: 0 TFRecord file(s)
  videos/: 0 TFRecord file(s)


### 4.1a Download from S3 / MinIO lakehouse

> **Runs when** `DATA_SOURCE = "s3_raw"` or `"s3_processed"`. Skip if using HuggingFace.

The Aegean AI lakehouse stores datasets in a MinIO-backed S3-compatible object store. The pouring dataset lives at:

```
s3://landing/tcc/pouring/raw/          ← TFRecord files
s3://landing/tcc/pouring/processed/    ← Pre-processed image folders
```

**Endpoints** (set `MINIO_ENDPOINT` in `.env`):

| Endpoint | Use case |
|----------|----------|
| `http://192.168.1.26:9000` | TrueNAS LAN direct (fastest for bulk download) |
| `https://s3.aegeanai.com` | Cloudflare Tunnel (remote access, valid TLS) |
| `http://localhost:29000` | Local Docker MinIO (development) |

**Credentials**: `MINIO_ACCESS_KEY` and `MINIO_SECRET_KEY` in `.env`.

In [8]:
def get_s3_client():
    """Create an S3 client for the MinIO lakehouse."""
    import boto3
    from botocore.config import Config

    return boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=S3_ACCESS_KEY,
        aws_secret_access_key=S3_SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
        use_ssl=S3_USE_SSL,
        verify=S3_USE_SSL,
    )


def s3_download_prefix(s3_client, bucket, prefix, local_dir):
    """Download all objects under an S3 prefix to a local directory."""
    local_dir = pathlib.Path(local_dir)
    paginator = s3_client.get_paginator("list_objects_v2")
    downloaded = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            rel_path = key[len(prefix):].lstrip("/")
            if not rel_path:
                continue
            dest = local_dir / rel_path
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists() and dest.stat().st_size == obj["Size"]:
                continue  # skip already-downloaded files
            s3_client.download_file(bucket, key, str(dest))
            downloaded += 1
            if downloaded % 100 == 0:
                print(f"  downloaded {downloaded} files...")

    print(f"  Done — {downloaded} new files downloaded to {local_dir}")
    return downloaded


if DATA_SOURCE == "s3_raw":
    s3 = get_s3_client()
    print(f"Downloading raw TFRecords from s3://{S3_BUCKET}/{S3_PREFIX}/raw/ ...")
    s3_download_prefix(s3, S3_BUCKET, f"{S3_PREFIX}/raw/", str(RAW_POURING_ROOT))

    for split_dir in sorted(RAW_POURING_ROOT.iterdir()):
        if split_dir.is_dir() and not split_dir.name.startswith("."):
            tfrecords = list(split_dir.glob("*.tfrecord*"))
            print(f"  {split_dir.name}/: {len(tfrecords)} TFRecord file(s)")

elif DATA_SOURCE == "s3_processed":
    s3 = get_s3_client()
    print(f"Downloading processed image folders from s3://{S3_BUCKET}/{S3_PREFIX}/processed/ ...")
    s3_download_prefix(s3, S3_BUCKET, f"{S3_PREFIX}/processed/", str(PROCESSED_POURING_ROOT))

    for split_dir in sorted(PROCESSED_POURING_ROOT.iterdir()):
        if split_dir.is_dir():
            videos = [d for d in split_dir.iterdir() if d.is_dir()]
            frames = sum(len(list(v.glob("*.png"))) for v in videos)
            print(f"  {split_dir.name}/: {len(videos)} videos, {frames} frames")

    print("\nProcessed data ready — no TFRecord conversion needed.")

else:
    print(f"Skipping S3 download — DATA_SOURCE={DATA_SOURCE!r}")

Skipping S3 download — DATA_SOURCE='huggingface'


In [9]:
# Step 3: Convert videos to image-folder layout
# HF pouring dataset has .mov videos under data/pouring/videos/{train,val,test}/
# videos_to_dataset extracts frames into layout expected by tcc.datasets.VideoDataset

import subprocess

if DATA_SOURCE == "s3_processed":
    print("Using pre-processed data from S3 — no conversion needed.")
else:
    VIDEO_INPUT_DIR = RAW_POURING_ROOT / "videos"
    print(f"Converting videos from {VIDEO_INPUT_DIR}...")
    subprocess.run(
        [sys.executable, "-m", "tcc.dataset_preparation.videos_to_dataset",
         "--input-dir", str(VIDEO_INPUT_DIR),
         "--output-dir", str(PROCESSED_POURING_ROOT / "pouring"),
         "--name", "pouring",
         "--fps", "15", "--width", "224", "--height", "224",
         "--file-pattern", "**/*.mov"],
        check=True)
    import os
    total = sum(len(f) for _, _, f in os.walk(PROCESSED_POURING_ROOT / "pouring"))
    print(f"Conversion complete: {total} files")


Converting videos from data/pouring/videos...


INFO: Processing video 1/470: box_to_clear0_real_view0


INFO: Processing video 2/470: box_to_clear0_real_view1


INFO: Processing video 3/470: box_to_clear1_fake_view0


INFO: Processing video 4/470: box_to_clear1_fake_view1


...

INFO: Processing video 468/470: whitesoda_to_clear0_real_view1


INFO: Processing video 469/470: whitewater_to_clear0_real_view0


INFO: Processing video 470/470: whitewater_to_clear0_real_view1


INFO: Dataset 'pouring' created at data/pouring_processed/pouring/pouring with 470 videos.


Conversion complete: 65571 files


In [10]:
# Step 4: Upload processed data to lakehouse (for future s3_processed runs)

if DATA_SOURCE != "s3_processed" and S3_ENDPOINT:
    try:
        s3 = get_s3_client()
        import os
        processed_root = PROCESSED_POURING_ROOT / "pouring"
        upload_count = 0
        for dirpath, dirnames, filenames in os.walk(processed_root):
            for fname in filenames:
                local_path = os.path.join(dirpath, fname)
                rel_path = os.path.relpath(local_path, PROCESSED_POURING_ROOT)
                s3_key = f"tcc/pouring_processed/{rel_path}"
                s3.upload_file(local_path, S3_BUCKET, s3_key)
                upload_count += 1
        print(f"Uploaded {upload_count} files to s3://{S3_BUCKET}/tcc/pouring_processed/")
    except Exception as e:
        print(f"S3 upload skipped: {e}")
else:
    print("Skipping S3 upload (data already from S3 or no endpoint configured)")


Uploaded 65571 files to s3://landing/tcc/pouring_processed/


### 4.2 Expected semantic phases

We will reason about pouring in terms of latent phases such as:

1. reach
2. grasp
3. lift / position
4. tilt
5. pour
6. retract / return

You do **not** need action labels for TCC training.  
These phase names are used only for qualitative interpretation of the learned representation.

## 5. Configuration and training

The current `aegean-ai/tcc` package provides:

- a typed configuration object
- an `alignment` algorithm corresponding to TCC
- a PyTorch training loop

The default configuration is useful to inspect first, because it tells us:

- training algorithm
- dataset name
- image size
- batch size
- embedding size
- checkpoint/logging schedule

In [11]:
from pprint import pprint

try:
    from tcc.config import get_default_config
    cfg = get_default_config()
    print(cfg)
except Exception as e:
    print("Could not import tcc yet:", repr(e))
    cfg = None

TCCConfig(logdir='/tmp/alignment_logs/', datasets=['pouring'], path_to_tfrecords='/tmp/%s_tfrecords/', training_algo='alignment', train=TrainConfig(max_iters=150000, batch_size=2, num_frames=20, visualize_interval=200), eval=EvalConfig(batch_size=2, num_frames=20, val_iters=20, tasks=['algo_loss', 'classification', 'kendalls_tau', 'event_completion', 'few_shot_classification'], frames_per_batch=25, kendalls_tau_stride=5, kendalls_tau_distance='sqeuclidean', classification_fractions=[0.1, 0.5, 1.0], few_shot_num_labeled=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], few_shot_num_episodes=50), model=ModelConfig(embedder_type='conv', base_model=BaseModelConfig(network='resnet50', layer='conv4_block3_out', train_base='only_bn'), conv_embedder=ConvEmbedderConfig(embedding_size=128, num_context_steps=2, conv_layers=[(256, 3, True), (256, 3, True)], fc_layers=[(256, True), (256, True)], capacity_scalar=2, flatten_method='max_pool', base_dropout_rate=0.0, base_dropout_spatial=False, fc_dropout_rate=0.1, dro

### 5.1 Utility: robust config editing

Research repositories evolve. Rather than assuming one exact config layout, we use helper functions that can set values safely if the corresponding fields exist.

This makes the notebook more resilient to small refactors of the dataclass hierarchy.

In [12]:
def set_if_exists(obj, path, value):
    parts = path.split(".")
    cur = obj
    for p in parts[:-1]:
        if not hasattr(cur, p):
            return False
        cur = getattr(cur, p)
    if hasattr(cur, parts[-1]):
        setattr(cur, parts[-1], value)
        return True
    return False

def get_if_exists(obj, path, default=None):
    parts = path.split(".")
    cur = obj
    for p in parts:
        if not hasattr(cur, p):
            return default
        cur = getattr(cur, p)
    return cur

def summarize_config(cfg):
    keys = [
        "training_algo",
        "datasets",
        "path_to_tfrecords",
        "logdir",
        "train.batch_size",
        "train.max_iters",
        "train.num_frames",
        "eval.batch_size",
        "model.embedder_type",
        "model.conv_embedder.embedding_size",
        "model.base_model.train_base",
        "optimizer.type",
        "optimizer.lr.initial_lr",
        "data.image_size",
        "data.frame_stride",
        "data.num_steps",
    ]
    rows = []
    for k in keys:
        rows.append((k, get_if_exists(cfg, k)))
    return rows

if cfg is not None:
    for k, v in summarize_config(cfg):
        print(f"{k:40s} {v}")

training_algo                            alignment
datasets                                 ['pouring']
path_to_tfrecords                        /tmp/%s_tfrecords/
logdir                                   /tmp/alignment_logs/
train.batch_size                         2
train.max_iters                          150000
train.num_frames                         20
eval.batch_size                          2
model.embedder_type                      conv
model.conv_embedder.embedding_size       128
model.base_model.train_base              only_bn
optimizer.type                           adam
optimizer.lr.initial_lr                  0.0001
data.image_size                          224
data.frame_stride                        15
data.num_steps                           2


### 5.2 Choose experiment settings

The assignment requires an embedding-dimension sweep:

- 32
- 64
- 128

We keep everything else as close as possible to the repo defaults so that the experiment isolates the representation bottleneck dimension.

In [13]:
EMBED_DIMS = [32, 64, 128]
EXPERIMENT_ROOT = pathlib.Path("runs_tutorial")
EXPERIMENT_ROOT.mkdir(exist_ok=True)

print("Experiments will be stored in:", EXPERIMENT_ROOT.resolve())
print("Embedding dims:", EMBED_DIMS)

Experiments will be stored in: /workspaces/tcc/runs_tutorial
Embedding dims: [32, 64, 128]


### 5.2a Weights & Biases logging (optional)

Training metrics (loss and learning rate) are logged to **TensorBoard** by default. To also stream them to **Weights & Biases**, set `WANDB_ENABLED = True` below.

Requirements:
- `pip install wandb` (already included in the `[notebooks]` extras)
- `WANDB_API_KEY` set in your environment (`.env` file or `wandb login`)

Set `WANDB_ENTITY` to your W&B team/user name, or leave empty for your default entity.

In [14]:
WANDB_ENABLED = True
WANDB_PROJECT = "tcc"
WANDB_ENTITY = ""  # your W&B team/user, or "" for default

if WANDB_ENABLED:
    try:
        import wandb
        print("wandb version:", wandb.__version__)
        print("W&B logging is ENABLED (project=%s)" % WANDB_PROJECT)
    except ImportError:
        print("wandb not installed — disabling W&B logging.")
        print("Install with: pip install wandb")
        WANDB_ENABLED = False
else:
    print("W&B logging is DISABLED. Set WANDB_ENABLED = True to enable.")

wandb version: 0.25.1
W&B logging is ENABLED (project=tcc)


### 5.3 Build a training config for one run

The training code in `src/tcc/train.py` expects a `TCCConfig`, and the default config already uses:

- `datasets: [pouring]`
- `training_algo: alignment`

We modify:

- embedding size
- log directory
- dataset root
- optionally `train.max_iters` for a shorter tutorial run

In [15]:
def make_run_config(embed_dim=128, max_iters=2000, logdir=None):
    from tcc.config import get_default_config

    cfg = get_default_config()

    set_if_exists(cfg, "training_algo", "alignment")
    set_if_exists(cfg, "datasets", ["pouring"])
    set_if_exists(cfg, "train.max_iters", max_iters)
    set_if_exists(cfg, "model.conv_embedder.embedding_size", embed_dim)

    ds_fmt = str((PROCESSED_POURING_ROOT / "%s").resolve())
    set_if_exists(cfg, "path_to_tfrecords", ds_fmt)

    if logdir is None:
        logdir = str((EXPERIMENT_ROOT / f"pouring_tcc_d{embed_dim}").resolve())
    set_if_exists(cfg, "logdir", logdir)

    # W&B logging
    set_if_exists(cfg, "logging.wandb_enabled", WANDB_ENABLED)
    set_if_exists(cfg, "logging.wandb_project", WANDB_PROJECT)
    set_if_exists(cfg, "logging.wandb_entity", WANDB_ENTITY)
    set_if_exists(cfg, "logging.wandb_run_name", f"pouring_d{embed_dim}")

    return cfg

try:
    demo_cfg = make_run_config(embed_dim=64, max_iters=500)
    for k, v in summarize_config(demo_cfg):
        print(f"{k:40s} {v}")
    print(f"{'logging.wandb_enabled':40s} {get_if_exists(demo_cfg, 'logging.wandb_enabled')}")
    print(f"{'logging.wandb_project':40s} {get_if_exists(demo_cfg, 'logging.wandb_project')}")
except Exception as e:
    print("Config construction failed:", repr(e))

training_algo                            alignment
datasets                                 ['pouring']
path_to_tfrecords                        /workspaces/tcc/data/pouring_processed/%s
logdir                                   /workspaces/tcc/runs_tutorial/pouring_tcc_d64
train.batch_size                         2
train.max_iters                          500
train.num_frames                         20
eval.batch_size                          2
model.embedder_type                      conv
model.conv_embedder.embedding_size       64
model.base_model.train_base              only_bn
optimizer.type                           adam
optimizer.lr.initial_lr                  0.0001
data.image_size                          224
data.frame_stride                        15
data.num_steps                           2
logging.wandb_enabled                    True
logging.wandb_project                    tcc


## 6. Training

The training loop in the repo is exposed through `tcc.train.train(cfg)`.

The logic is:

1. instantiate the algorithm corresponding to `cfg.training_algo`
2. build the dataset loader
3. optimize the alignment loss
4. save checkpoints in `cfg.logdir`

In [16]:
def run_training(cfg):
    from tcc.train import train
    print("Starting training with logdir:", cfg.logdir)
    train(cfg)

# Debug run — short iteration count to verify the pipeline works
cfg_debug = make_run_config(embed_dim=32, max_iters=50)
run_training(cfg_debug)

Starting training with logdir: /workspaces/tcc/runs_tutorial/pouring_tcc_d32


wandb: Tracking run with wandb version 0.25.1


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /workspaces/tcc/runs_tutorial/pouring_tcc_d32/wandb/offline-run-20260315_141506-ecln93yo


wandb: WARNING URL not available in offline run


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/vscode/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


...

 63%|██████▎   | 62.0M/97.8M [00:00<00:00, 87.8MB/s]

 73%|███████▎  | 71.1M/97.8M [00:00<00:00, 89.9MB/s]

 82%|████████▏ | 80.5M/97.8M [00:00<00:00, 92.1MB/s]

 92%|█████████▏| 89.6M/97.8M [00:01<00:00, 92.9MB/s]

100%|██████████| 97.8M/97.8M [00:01<00:00, 91.1MB/s]

### 6.1 Full assignment runs

Run three experiments:

- $D=32$
- $D=64$
- $D=128$

In [ ]:
for d in EMBED_DIMS:
    cfg_run = make_run_config(embed_dim=d, max_iters=5000)
    run_training(cfg_run)

## 7. Loading checkpoints and extracting embeddings

The repo provides the pieces we need:

- `get_algo(...)` to instantiate the TCC algorithm
- checkpoint loading utilities from `tcc.train`
- embedding extraction utilities from `tcc.evaluate`

In [ ]:
import torch
from pathlib import Path

def latest_checkpoint(logdir):
    candidates = sorted(Path(logdir).glob("checkpoint_*.pt"))
    if not candidates:
        return None
    return str(candidates[-1])

def load_trained_algo(cfg, checkpoint_path=None):
    from tcc.algos.registry import get_algo
    from tcc.train import load_checkpoint

    algo = get_algo(cfg.training_algo, cfg=cfg)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    algo = algo.to(device)

    if checkpoint_path is None:
        checkpoint_path = latest_checkpoint(cfg.logdir)

    if checkpoint_path is None:
        raise FileNotFoundError(f"No checkpoint found in {cfg.logdir}")

    _ = load_checkpoint(checkpoint_path, algo, optimizer=None)
    algo.eval()
    return algo, device, checkpoint_path

### 7.1 Build the evaluation dataloader

The repo training code internally converts the top-level config into a `DataConfig`.  
We reuse the same helper if available; otherwise we build the `DataConfig` manually.

In [ ]:
def make_eval_dataloader(cfg, split="val", mode="eval"):
    """Build an evaluation dataloader from the top-level config.

    Uses the repo's internal _build_data_config helper.  If that helper
    is missing or its signature has changed, the call fails loudly so
    you know the notebook and repo are out of sync.
    """
    from tcc.datasets import create_dataset

    try:
        from tcc.train import _build_data_config
    except ImportError:
        raise ImportError(
            "Cannot import _build_data_config from tcc.train. "
            "The aegean-ai/tcc repo API may have changed. "
            "Check the repo README for the current evaluation interface."
        )

    data_cfg = _build_data_config(cfg)
    loader = create_dataset(split=split, mode=mode, config=data_cfg)
    return loader

In [ ]:
def extract_embeddings_for_run(cfg, split="val", max_embs=0):
    """Extract embeddings from a trained checkpoint.

    Args:
        cfg: TCCConfig for the run.
        split: dataset split to evaluate ("val" or "train").
        max_embs: maximum number of video embeddings to extract.
                  0 means extract all available videos (no limit).
    """
    from tcc.evaluate import get_embeddings_dataset

    algo, device, checkpoint_path = load_trained_algo(cfg)
    loader = make_eval_dataloader(cfg, split=split, mode="eval")
    bundle = get_embeddings_dataset(algo, loader, device=device, max_embs=max_embs)

    print("Loaded checkpoint:", checkpoint_path)
    print("Videos:", len(bundle["embeddings_list"]))
    print("Flat embeddings shape:", bundle["embeddings"].shape)
    return bundle

# Example:
# cfg64 = make_run_config(embed_dim=64, max_iters=5000)
# emb_bundle = extract_embeddings_for_run(cfg64, split="val")

## 8. Representation diagnostics

Now we test the main scientific claim:

> Do embeddings organize frames by **task phase**?

We use two projection methods and two diagnostic approaches:

1. **PCA** — linear projection preserving global variance; fast and deterministic
2. **UMAP** — nonlinear projection revealing manifold structure; better for fine-grained phase separation

For each, we produce:

- **single-video trajectory plots** colored by time
- **cross-video overlays** in a shared projection space (joint fit, so coordinates are comparable)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

def project_pca(Z):
    return PCA(n_components=2).fit_transform(Z)

def project_umap(Z, seed=0):
    if not HAS_UMAP:
        raise ImportError("Install umap-learn: pip install umap-learn")
    return umap.UMAP(n_components=2, random_state=seed).fit_transform(Z)

def project(Z, method="umap", seed=0):
    if method == "pca":
        return project_pca(Z)
    if method == "umap":
        return project_umap(Z, seed=seed)
    raise ValueError(f"Unknown method: {method}. Use 'pca' or 'umap'.")

In [ ]:
def plot_single_trajectory(Z, method="umap", title=None):
    Y = project(Z, method=method)
    t = np.arange(len(Y))

    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Y[:, 0], Y[:, 1], c=t, s=12)
    plt.colorbar(sc, label="time")
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.title(title or f"{method.upper()} trajectory")
    plt.tight_layout()
    plt.show()

def plot_multiple_trajectories(embeddings_list, names=None, method="umap", max_videos=6):
    """Plot cross-video trajectories in a shared projection space.

    All video embeddings are concatenated, projected once, then split
    back so that the 2D coordinates are comparable across videos.
    """
    n = min(max_videos, len(embeddings_list))
    selected = embeddings_list[:n]
    lengths = [len(Z) for Z in selected]
    Z_all = np.concatenate(selected, axis=0)

    Y_all = project(Z_all, method=method, seed=0)

    splits = np.cumsum(lengths[:-1])
    Y_per_video = np.split(Y_all, splits)

    plt.figure(figsize=(7, 6))
    for i, Y in enumerate(Y_per_video):
        label = names[i] if names else f"video_{i}"
        plt.plot(Y[:, 0], Y[:, 1], alpha=0.8, label=label)
    plt.xlabel("component 1")
    plt.ylabel("component 2")
    plt.title(f"{method.upper()} cross-video trajectories (joint projection)")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
emb_bundle = extract_embeddings_for_run(make_run_config(embed_dim=64, max_iters=5000), split="val")
Z0 = emb_bundle["embeddings_list"][0]
plot_single_trajectory(Z0, method="pca", title="PCA trajectory (D=64)")
plot_single_trajectory(Z0, method="umap", title="UMAP trajectory (D=64)")
plot_multiple_trajectories(emb_bundle["embeddings_list"], emb_bundle.get("names"), method="umap")

## 9. Temporal segmentation from embedding geometry

This section operationalizes the claim that the embedding has learned latent phase.

We use two complementary segmentation strategies:

### 9.1 Change-point detection

If the representation changes rapidly at phase transitions, then
$$
d_t = \|z_t - z_{t-1}\|
$$
should spike near boundaries. This is a **boundary-detection** approach — it finds *where* phase transitions occur without assigning cluster labels.

### 9.2 KMeans clustering in the native embedding space

If the embedding clusters by phase, KMeans should recover coarse phase labels. We use $k=6$ to match the six expected pouring phases (reach, grasp, lift, tilt, pour, retract). Experiment with different $k$ values to test sensitivity.

In [ ]:
from sklearn.cluster import KMeans

def change_point_scores(Z):
    d = np.linalg.norm(Z[1:] - Z[:-1], axis=1)
    d = np.concatenate([[0.0], d])
    return d

def detect_boundaries(d, threshold_quantile=0.98, min_gap=10):
    thr = float(np.quantile(d, threshold_quantile))
    idx = np.where(d >= thr)[0].tolist()

    kept = []
    last = -10**9
    for i in idx:
        if i - last >= min_gap:
            kept.append(i)
            last = i
    return kept, thr

def cluster_kmeans(Z, k=6, seed=0):
    """KMeans with k=6 matching the six expected pouring phases."""
    return KMeans(n_clusters=k, random_state=seed, n_init="auto").fit_predict(Z)

In [ ]:
def plot_segmentation(Z, labels=None, boundaries=None, title="Segmentation"):
    d = change_point_scores(Z)
    T = len(Z)

    plt.figure(figsize=(10, 3))
    plt.plot(np.arange(T), d)
    if boundaries is not None:
        for b in boundaries:
            plt.axvline(b, linestyle="--")
    plt.title(title + " — change-point score")
    plt.xlabel("frame index")
    plt.ylabel(r"$\|z_t-z_{t-1}\|$")
    plt.tight_layout()
    plt.show()

    if labels is not None:
        plt.figure(figsize=(10, 2))
        plt.plot(np.arange(T), labels, drawstyle="steps-mid")
        if boundaries is not None:
            for b in boundaries:
                plt.axvline(b, linestyle="--")
        plt.title(title + " — cluster labels over time")
        plt.xlabel("frame index")
        plt.ylabel("cluster")
        plt.tight_layout()
        plt.show()

In [ ]:
Z = emb_bundle["embeddings_list"][0]
d = change_point_scores(Z)
boundaries, thr = detect_boundaries(d, threshold_quantile=0.98, min_gap=8)
labels_km = cluster_kmeans(Z, k=6)

plot_segmentation(Z, labels=labels_km, boundaries=boundaries, title="KMeans on native embeddings (D=64)")

## 10. Embedding dimension sweep

The assignment asks you to compare:

- $D=32$
- $D=64$
- $D=128$

This matters because the embedding dimension controls the trade-off between:

- **compression**
- **expressiveness**
- **ease of clustering**
- **risk of overfitting appearance rather than phase**

In [ ]:
def run_full_analysis_for_dimension(embed_dim, max_iters=5000, split="val",
                                    max_embs=0, num_videos_to_analyze=3):
    """Run the complete analysis pipeline for one embedding dimension.

    Analyzes up to num_videos_to_analyze videos (not just the first one)
    to ensure results are not artifacts of a single video.
    """
    cfg = make_run_config(embed_dim=embed_dim, max_iters=max_iters)
    bundle = extract_embeddings_for_run(cfg, split=split, max_embs=max_embs)

    print(f"\n===== Dimension {embed_dim} =====")
    print("Number of videos:", len(bundle["embeddings_list"]))
    print("Flat embedding matrix:", bundle["embeddings"].shape)

    if len(bundle["embeddings_list"]) == 0:
        print("No embeddings found.")
        return cfg, bundle

    n_analyze = min(num_videos_to_analyze, len(bundle["embeddings_list"]))

    for vid_idx in range(n_analyze):
        Z = bundle["embeddings_list"][vid_idx]
        vid_name = bundle["names"][vid_idx] if "names" in bundle else f"video_{vid_idx}"
        tag = f"D={embed_dim}, {vid_name}"

        plot_single_trajectory(Z, method="pca", title=f"PCA trajectory ({tag})")
        if HAS_UMAP:
            plot_single_trajectory(Z, method="umap", title=f"UMAP trajectory ({tag})")

        d = change_point_scores(Z)
        boundaries, thr = detect_boundaries(d, threshold_quantile=0.98, min_gap=8)
        labels_km = cluster_kmeans(Z, k=6)
        plot_segmentation(Z, labels=labels_km, boundaries=boundaries,
                          title=f"KMeans k=6 ({tag})")

    # Cross-video overlay (joint projection)
    if HAS_UMAP and len(bundle["embeddings_list"]) > 1:
        plot_multiple_trajectories(bundle["embeddings_list"], bundle.get("names"),
                                   method="umap")

    return cfg, bundle

# Run full analysis for all three embedding dimensions
results = {}
for d in EMBED_DIMS:
    cfg_d, bundle_d = run_full_analysis_for_dimension(d, max_iters=5000)
    results[d] = (cfg_d, bundle_d)

## 11. Write-up questions

### Q1. TCN vs TCC
Explain, in your own words, the evolution from TCN to TCC. Include the role of the soft nearest-neighbor formulation in making cycle consistency differentiable.

### Q2. Does the learned representation encode phase?
Use your PCA and UMAP plots to justify a claim. Compare single-video trajectories with cross-video overlays.

### Q3. How well does segmentation recover phase structure?
Compare change-point detection and KMeans clustering. Do the detected boundaries align with qualitative phase transitions? Does varying $k$ change the story?

### Q4. What failure modes remain?
Examples:

- appearance variation dominating phase
- pauses causing over-segmentation
- self-similar frames across non-adjacent stages
- collapse of distinct phases into one cluster

## 12. Final checklist

Before submitting, verify that you have:

- trained at least one real TCC run on pouring
- extracted embeddings from a saved checkpoint
- produced PCA and UMAP trajectory plots for multiple videos
- produced a cross-video overlay using joint projection
- run both change-point detection and KMeans segmentation
- compared $D=32,64,128$
- written answers to all four questions (Q1–Q4) in inline markdown cells, with all supporting figures embedded in the notebook

That completes the assignment.